# Crop productivity correction — CSV → GeoParquet

Prepare Food-Security salinity-corrected yield results for the Global Coastal Atlas STAC.

Workflow (same section style as `11b_Salinity.ipynb`):

1. Configure paths
2. Read corrected-yield CSV
3. Join administrative polygons (`province_fwzone.shp`)
4. Write GeoParquet under `stac_folder/crop_productivity_correction/`
5. Write / refresh metadata JSON for `scripts/13_crop_productivity_correction.ipynb`


### Import packages


In [11]:
import json
from pathlib import Path

import geopandas as gpd
import pandas as pd

from coclicodata.drive_config import p_drive


### Define drive paths


In [12]:
processed_data_dir = p_drive.joinpath(
    r"P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\crop_productivity_correction"
)

csv_path = Path(
    r"C:\Users\fuentesm\OneDrive - Stichting Deltares\International Delta Toolset - Salinity\Vietnam\results\corrected-yield-base-stac.csv"
)

# Same admin units as Food-Security STAC yield CSV (Name == area_map_name)
provinces_shp = Path(
    r"C:\Users\fuentesm\OneDrive - Stichting Deltares\International Delta Toolset - Salinity\Vietnam\province_fwzone.shp"
)

scenario = "baseline"
parquet_dir = processed_data_dir.joinpath("parquets", scenario)
parquet_name = "corrected_yield.parquet"
metadata_path = processed_data_dir.joinpath(
    "metadata_crop_productivity_correction.json"
)

print("Output dir:", processed_data_dir)
print("CSV:", csv_path, "exists:", csv_path.is_file())
print("Provinces:", provinces_shp, "exists:", provinces_shp.is_file())


Output dir: P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\crop_productivity_correction
CSV: C:\Users\fuentesm\OneDrive - Stichting Deltares\International Delta Toolset - Salinity\Vietnam\results\corrected-yield-base-stac.csv exists: True
Provinces: C:\Users\fuentesm\OneDrive - Stichting Deltares\International Delta Toolset - Salinity\Vietnam\province_fwzone.shp exists: True


### Read raw data


In [13]:
df = pd.read_csv(csv_path)
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

print("Rows:", len(df))
print("Columns:", list(df.columns))
print("Years:", sorted(df["year"].dropna().unique().tolist()))
print("Areas:", df["area_map_name"].nunique())
df.head()


Rows: 228
Columns: ['area_map_name', 'crop_name', 'crop_name_fao', 'salinity', 'yield', 'hectares', 'year', 'a', 'b', 'corrected_yield', 'corrected_yield_pp', 'comment', 'FTE']
Years: [2014, 2015, 2016]
Areas: 19


,area_map_name,crop_name,crop_name_fao,salinity,yield,hectares,year,a,b,corrected_yield,corrected_yield_pp,comment,FTE
0,An Giang_,AutumnWinter (ha),"Rice, paddy",0.000000e+00,0.0,0.0,2016,0.0,0.0,0.0,0.0,No correction needed,0.0
1,An Giang_,WinterSpring (ha),"Rice, paddy",1.683842e-40,0.0,0.0,2015,3.0,12.0,0.0,0.0,NaN,0.0
2,An Giang_,SummerAutumn (ha),"Rice, paddy",0.000000e+00,0.0,0.0,2015,0.0,0.0,0.0,0.0,No correction needed,0.0
3,An Giang_,SummerAutumn (ha),"Rice, paddy",0.000000e+00,0.0,0.0,2014,0.0,0.0,0.0,0.0,No correction needed,0.0
4,An Giang_,AutumnWinter (ha),"Rice, paddy",0.000000e+00,0.0,0.0,2015,0.0,0.0,0.0,0.0,No correction needed,0.0


### Check CF compliancy

Not applicable for tabular CSV / GeoParquet (same spirit as geotiff CF checks in `11b`).


In [14]:
# Not implemented for tabular GeoParquet


### Make alterations (join geometry)

CSV has no geometry. Join `province_fwzone.shp` on `Name` = `area_map_name`, then reproject to EPSG:4326 for STAC.


In [15]:
gdf_prov = gpd.read_file(provinces_shp)
if "Name" not in gdf_prov.columns:
    raise KeyError(
        f"Expected Name column in {provinces_shp}, got {list(gdf_prov.columns)}"
    )

gdf = gdf_prov.merge(
    df,
    left_on="Name",
    right_on="area_map_name",
    how="inner",
    validate="one_to_many",
)

csv_areas = set(df["area_map_name"].astype(str))
matched = set(gdf["area_map_name"].astype(str))
missing = sorted(csv_areas - matched)
if missing:
    raise ValueError(
        f"{len(missing)} area_map_name values did not match province Name: {missing[:10]}"
    )

if gdf.crs is None:
    raise ValueError("Provinces shapefile has no CRS")
gdf = gdf.to_crs(epsg=4326)

print("Joined rows:", len(gdf), "(csv rows:", len(df), ")")
print("CRS:", gdf.crs)
print("Bounds (4326):", tuple(gdf.total_bounds))
gdf.head(2)


Joined rows: 228 (csv rows: 228 )
CRS: EPSG:4326
Bounds (4326): (np.float64(103.41797806300008), np.float64(8.380953215000092), np.float64(106.82566220200007), np.float64(11.03326806100009))


,Name,OBJECTID,zone,crop_affec,geometry,area_map_name,crop_name,crop_name_fao,salinity,yield,hectares,year,a,b,corrected_yield,corrected_yield_pp,comment,FTE
0,Bac Lieu_S0001,1,Fresh water,NaN,"POLYGON ((105.54307 9.45208, 105.55365 9.44034...",Bac Lieu_S0001,MUA (ha),"Rice, paddy",0.0,1290364.5,318.1372,2015,0.0,0.0,1290364.5,3.883137e+05,No correction needed,413.578365
1,Bac Lieu_S0001,1,Fresh water,NaN,"POLYGON ((105.54307 9.45208, 105.55365 9.44034...",Bac Lieu_S0001,AutumnWinter (ha),"Rice, paddy",0.0,53326036.0,10999.5940,2014,0.0,0.0,53326036.0,1.604758e+07,No correction needed,17091.678205


### Write data to GeoParquet


#### Single GeoParquet (baseline)


In [16]:
parquet_dir.mkdir(parents=True, exist_ok=True)
out_path = parquet_dir.joinpath(parquet_name)

gdf.to_parquet(out_path, index=False)
print("Wrote:", out_path)
print("Size MB:", round(out_path.stat().st_size / 1e6, 3))


Wrote: P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\crop_productivity_correction\parquets\baseline\corrected_yield.parquet
Size MB: 0.197


In [17]:
# Round-trip check
gdf_check = gpd.read_parquet(out_path)
assert len(gdf_check) == len(gdf)
assert gdf_check.geometry.notna().all()
print("Round-trip OK:", len(gdf_check), "features,", gdf_check.crs)
gdf_check.head(2)


Round-trip OK: 228 features, {"$schema": "https://proj.org/schemas/v0.7/projjson.schema.json", "type": "GeographicCRS", "name": "WGS 84", "datum_ensemble": {"name": "World Geodetic System 1984 ensemble", "members": [{"name": "World Geodetic System 1984 (Transit)"}, {"name": "World Geodetic System 1984 (G730)"}, {"name": "World Geodetic System 1984 (G873)"}, {"name": "World Geodetic System 1984 (G1150)"}, {"name": "World Geodetic System 1984 (G1674)"}, {"name": "World Geodetic System 1984 (G1762)"}, {"name": "World Geodetic System 1984 (G2139)"}, {"name": "World Geodetic System 1984 (G2296)"}], "ellipsoid": {"name": "WGS 84", "semi_major_axis": 6378137, "inverse_flattening": 298.257223563}, "accuracy": "2.0", "id": {"authority": "EPSG", "code": 6326}}, "coordinate_system": {"subtype": "ellipsoidal", "axis": [{"name": "Geodetic latitude", "abbreviation": "Lat", "direction": "north", "unit": "degree"}, {"name": "Geodetic longitude", "abbreviation": "Lon", "direction": "east", "unit": "deg

,Name,OBJECTID,zone,crop_affec,geometry,area_map_name,crop_name,crop_name_fao,salinity,yield,hectares,year,a,b,corrected_yield,corrected_yield_pp,comment,FTE
0,Bac Lieu_S0001,1,Fresh water,NaN,"POLYGON ((105.54307 9.45208, 105.55365 9.44034...",Bac Lieu_S0001,MUA (ha),"Rice, paddy",0.0,1290364.5,318.1372,2015,0.0,0.0,1290364.5,3.883137e+05,No correction needed,413.578365
1,Bac Lieu_S0001,1,Fresh water,NaN,"POLYGON ((105.54307 9.45208, 105.55365 9.44034...",Bac Lieu_S0001,AutumnWinter (ha),"Rice, paddy",0.0,53326036.0,10999.5940,2014,0.0,0.0,53326036.0,1.604758e+07,No correction needed,17091.678205


#### Metadata JSON for STAC publish script


In [18]:
metadata = json.loads('''{"TITLE": "Crop productivity correction (salinity)", "TITLE_ABBREVIATION": "CropProdCorr", "DESCRIPTION": "Salinity-corrected crop production and related labour indicators for Vietnam Mekong Delta provinces / freshwater zones. Tabular results from Food-Security salinity correction joined to administrative polygons.", "SHORT_DESCRIPTION": "Salinity-corrected crop yield by area and year (GeoParquet).", "INSTITUTION": "Deltares", "PROVIDERS": {"name": "Deltares", "url": "https://www.deltares.nl", "roles": "provider", "description": "International Delta Platform / Food Security"}, "HISTORY": ["Deltares", "Food-Security salinity_correction"], "MEDIA_TYPE": "application/vnd.apache.parquet", "DATA_MODEL": "geoparquet_table", "DIMENSIONS": ["year", "area_map_name"], "SPATIAL_EXTENT": [104.532771, 8.580031, 107.024243, 11.244372], "TEMPORAL_EXTENT": ["2014-01-01T00:00:00", "2016-12-31T00:00:00"], "LICENSE": "Creative Commons Attribution 4.0", "AUTHOR": "Deltares", "KEYWORDS": ["crop productivity", "salinity correction", "Vietnam", "Mekong", "food security", "GeoParquet"], "TAGS": ["crop", "salinity", "yield", "parquet"], "CITATION": "Deltares International Delta Platform \u2014 Food Security salinity correction", "DOI": "", "LONG_NAME": "Corrected crop yield", "UNITS": "t; ha; FTE", "COMMENT": "Geometry joined from province_fwzone.shp on Name = area_map_name. Source CSV: corrected-yield-base-stac.csv", "CRS": "EPSG:4326", "ITEM_BBOX_CRS": "EPSG:4326", "SPATIAL_RESOLUTION": null, "NODATA": null, "DATA_TYPE": "float64", "COLLECTION_ID": "crop_productivity_correction", "WMS_DATASET": "crop_productivity_correction"}''')

# Refresh temporal / spatial extent from the GeoParquet
years = sorted(int(y) for y in gdf["year"].dropna().unique())
metadata["TEMPORAL_EXTENT"] = [
    f"{years[0]}-01-01T00:00:00",
    f"{years[-1]}-12-31T00:00:00",
]
minx, miny, maxx, maxy = map(float, gdf.total_bounds)
metadata["SPATIAL_EXTENT"] = [minx, miny, maxx, maxy]

processed_data_dir.mkdir(parents=True, exist_ok=True)
with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)
print("Wrote metadata:", metadata_path)
print("COLLECTION_ID:", metadata["COLLECTION_ID"])
print("TEMPORAL_EXTENT:", metadata["TEMPORAL_EXTENT"])
print("SPATIAL_EXTENT:", metadata["SPATIAL_EXTENT"])


Wrote metadata: P:\11211454-002-idt\IDP\Vietnam\Mekong\salinity_mekong\preprocessed_outputs\stac_folder\crop_productivity_correction\metadata_crop_productivity_correction.json
COLLECTION_ID: crop_productivity_correction
TEMPORAL_EXTENT: ['2014-01-01T00:00:00', '2016-12-31T00:00:00']
SPATIAL_EXTENT: [103.41797806300008, 8.380953215000092, 106.82566220200007, 11.03326806100009]


#### Multiple scenarios (optional)

Extend later with additional CSVs (`corrected-yield-cc45y.csv`, …) → `parquets/{scenario}/corrected_yield.parquet`.


In [19]:
# Placeholder for multi-scenario loop
# scenario_csvs = {
#     "baseline": csv_path,
# }
